## Paper Auto-Classifier 📚
### 1) Importar Librerias 

In [29]:
#Asegurate de tener activado el entorno de programación con el interprete python 3.11
# (En terminal) 
# pip install langchain_huggingface
# pip install tqdm

import pandas as pd 
import numpy as np
from LLM import Clasificador
from tqdm import tqdm

Unificar .CSV

In [ ]:
# import os
# import pandas as pd

# # Carpeta donde están los archivos
# carpeta = r"E:\LLMzCor\LLMzCor.github.io\WebScraping\Abstracts"

# # Lista de nombres de archivos que quieres concatenar
# nombres_archivos = [
#     "PMID_Hp.csv",
#     "PMID_Sh.csv",
#     "PMID_Kp.csv",
   
#     # agrega aquí los nombres que desees
# ]

# # Construye las rutas completas
# rutas = [os.path.join(carpeta, nombre) for nombre in nombres_archivos]

# # Lee y concatena los archivos
# df_list = [pd.read_csv(ruta) for ruta in rutas]
# df_concat = pd.concat(df_list, ignore_index=True)

# # Guarda el resultado si lo deseas
# Output_file_name = "compilado_Hp-Sh-Kp.csv"  # Cambia aquí el nombre como desees
# df_concat.to_csv(os.path.join(carpeta, Output_file_name ), index=False, encoding="utf-8-sig")

# print(f"Archivo guardado como: {Output_file_name}", rutas)
# print("Shape final:", df_concat.shape)

Archivo guardado como: compilado_Hp-Sh-Kp.csv ['E:\\LLMzCor\\LLMzCor.github.io\\WebScraping\\Abstracts\\PMID_Hp.csv', 'E:\\LLMzCor\\LLMzCor.github.io\\WebScraping\\Abstracts\\PMID_Sh.csv', 'E:\\LLMzCor\\LLMzCor.github.io\\WebScraping\\Abstracts\\PMID_Kp.csv']
Shape final: (37460, 2)


### 2) Carga BD de Bacterias 🦠🧫👨‍🔬👩‍🔬

In [30]:
#Modifica la ruta de la base de datos
#Selecona la pesataña del excel
file_path = r"E:\LLMzCor\LLMzCor.github.io\WebScraping\Abstracts\PMID_Ng.csv"
# sheet = "Test"
# usecols= [
#     'PMID',
#     'Title',
#     'Abstract',
#     #'Estado',
#     '1) Antimicrobial Resistance stain',
#     '2) New treatment',
#     '3) Immunization',
#     'Human_summary',
#     # 'Publication Year',
#     # 'Journal/Book',
#     # 'Alerta'
# ]
df = pd.read_csv(
    file_path,
    # sheet_name=sheet,
    # usecols=usecols, 
    index_col=0
)
df.shape

(3740, 1)

In [31]:
for col in [
    'Title',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary',
]:
    if col not in df.columns: 
        df[col] = np.nan

#### Aplica filtros de Excluir (Opcional)

In [32]:
#Filtro los excluidos
# df = df[df['Estado']!="Excluir"]

# df_shape = df.shape

# #Calculo cuantos paper se excluyeron
# resta = df .shape[0]- df.shape[0]
# print("df Shape ", df_shape, " depués del filtro ", resta)

df.head()

,Title,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,
29430011,The host-adapted human pathogen Neisseria gono...,NaN,NaN,NaN,NaN
28991324,No abstract available,NaN,NaN,NaN,NaN
31119616,Gonorrhea and antimicrobial resistance (AMR) i...,NaN,NaN,NaN,NaN
31802195,Neisseria gonorrhoeae is an etiologic agent of...,NaN,NaN,NaN,NaN
28754299,Neisseria gonorrhoeae is the agent of gonorrhe...,NaN,NaN,NaN,NaN


#### _____________2.b) Carga de BD de Tratamientos ⚗️🧪👨‍🔬(Opcional)

In [7]:
#Caraga la base de datos de los tratamientos (tx) para cada bacteria
# treatment_path = r"E:\PaperLLM\LLMzCor.github.io\DBs\Treatment_2017.xlsx"
# tx_df = pd.read_excel(io= treatment_path, index_col=0)
# #tx_df.head()

# #Selecciona los tratamiento de primera eleccion utilizados hasta 2017
# Bacteria = "Chlamydia_trachomatis" 
# treatment= tx_df.loc[Bacteria,"First-line treatment until 2017"]
# print(treatment)

### 3) LLM Funciones ⚠️ Importante Ejecutar‼️

In [33]:
clasificador=Clasificador()

In [34]:
def _transformacion_binario_val(col):
    if (col=="Yes") | (col==1):
        return 1
    elif (col=="No") | (col==0):
        return 0
def _transformacion_binario_df(df):
    df[['1) Antimicrobial Resistance stain',
        '2) New treatment','3) Immunization']] = df[['1) Antimicrobial Resistance stain',
                                                     '2) New treatment',
                                                     '3) Immunization']].applymap(_transformacion_binario_val)
    return df



def ask_llm(df_,partition=0):
    df=df_.copy()
    df=_transformacion_binario_df(df)
    df["ai_label"]=np.nan
    df["ai_summary"]=np.nan
    if partition==0:
        for pmid in df.index:
            try:
                response = clasificador.clasificacion(df.loc[pmid,"Abstract"])
                df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df.loc[:pmid].iloc[:-1]
        return df
    else:
        print("aca")
        df_ptit=df.iloc[:partition,:]
        for pmid in df_ptit.index:
            try:
                response = clasificador.clasificacion(df_ptit.loc[pmid,"Abstract"])
                df_ptit.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df_ptit.loc[:pmid].iloc[:-1]
        return df_ptit

def _evaluate(row):
    values = row[["1) Antimicrobial Resistance stain","2) New treatment","3) Immunization"]].values
    ai_opcion=int(row["ai_label"])-1
    print(f"values={values}")
    print(f"ai_opcion{ai_opcion}")
    if (values.sum()==0) & (ai_opcion==3):
        return 1
    elif (values.sum()==0) & (ai_opcion<3):
        return 0
    elif (values.sum()>0) & (ai_opcion==3):
        return 0
    elif values[ai_opcion]>0:

        return 1
    else:
        return 0

def evaluacion_score(df,partition=0):
    if partition ==0:
        n=df.shape[0]
        scores = df.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    else:
        df_ptit=df.iloc[:10,:]
        n=df_ptit.shape[0]
        scores=df_ptit.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    return final_score

### _______________________Prueba clasificando un solo paper (Opcional)

In [18]:

paper=df.loc[22203235, "Abstract"]
#paper=df.loc[22525317, "Title"]
clasificador.clasificacion(paper)

['4',
 ' "The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on the mechanisms of Chlamydia pneumoniae invasion into nonphagocytic epithelial cells, specifically the role of cholesterol, sphingomyelin, and certain receptors in this process."']

### 4) Ask LLM to classify df 

---
Abreviatura dataframes de bacterias:
+ df --> Chlamydia trachomatis
+ df_Cd --> Clostridium difficile
+ df_Hi --> Haemopilus influenzae
+ df_Kp --> Klebsiella pneumoniae
+ df_Ng --> Neisseria gonorrhoeae
+ df_Sh --> Shigela spp
+ df_Rk --> Ricketttsia
---

### 4b) Ask df_ptits Particionado

### ----------Automatización de clasificaciones ----------

In [36]:
import os
from datetime import datetime

# Define el tamaño de cada partición
batch_size = 10
df_list = []
file_base = os.path.splitext(os.path.basename(file_path))[0]

try:
    # Itera sobre el DataFrame en bloques de batch_size
    for start in range(0, df.shape[0], batch_size):
        end = start + batch_size
        df_ptit = df.iloc[start:end, :]
        print(f"Procesando filas {start} a {end-1}")
        df_classified = ask_llm(df_ptit)
        df_list.append(df_classified)

    # Une todos los resultados en un solo DataFrame
    df_Cta = pd.concat(df_list, ignore_index=False)
    print("Shape final:", df_Cta.shape)
    df_Cta.head()
except Exception as e:
    if df_list :
        df_partial = pd.concat(df_list, ignore_index=False)
        now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        partial_path = f"{file_base}_partial_{now}.csv"
        df_partial.to_csv(partial_path, encoding='utf-8-sig')
        print(f"Guardado parcial en: {partial_path}")
        pmid = df_partial.index[-1]
    print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
    df_Cta = df_classified.loc[:pmid].iloc[:-1]
    print("Shape final:", df_Cta.shape)
    raise

Procesando filas 0 a 9
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 29430011
Procesando filas 10 a 19
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 28645177
Procesando filas 20 a 29
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 34094619
Procesando filas 30 a 39
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 32530860
Procesando filas 40 a 49
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 29883770
Procesando filas 50 a 59
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 25480491
Procesando filas 60 a 69
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 33590810
Procesando filas 70 a 79
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 31340021
Procesando filas 80 a 89
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 33367585
Procesando filas 90 a 99
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 27185804
Pr

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.


Se corto el proceso del LLM por este motivo : 'Abstract' en el id 32441045
Procesando filas 180 a 189
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 27758727
Procesando filas 190 a 199
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 29373697
Procesando filas 200 a 209
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 30321622
Procesando filas 210 a 219
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 27736753
Procesando filas 220 a 229
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 32318263
Procesando filas 230 a 239
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 23680903
Procesando filas 240 a 249
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 22214779
Procesando filas 250 a 259
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 30808445
Procesando filas 260 a 269
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 31955892
Procesa

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.


Se corto el proceso del LLM por este motivo : 'Abstract' en el id 32964934
Procesando filas 530 a 539
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 22807549
Procesando filas 540 a 549
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 30275084
Procesando filas 550 a 559
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 23186215
Procesando filas 560 a 569
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 24435165
Procesando filas 570 a 579
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 27080231
Procesando filas 580 a 589
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 26244506
Procesando filas 590 a 599
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 31119624
Procesando filas 600 a 609
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 31860664
Procesando filas 610 a 619
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 27494921
Procesa

...

Se corto el proceso del LLM por este motivo : 'Abstract' en el id 34223105
Procesando filas 2810 a 2819
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 22877601
Procesando filas 2820 a 2829
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 29573486
Procesando filas 2830 a 2839
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 32214342
Procesando filas 2840 a 2849
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 34706064
Procesando filas 2850 a 2859
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 35154028
Procesando filas 2860 a 2869
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 25768859
Procesando filas 2870 a 2879
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 24275271
Procesando filas 2880 a 2889
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 26442504
Procesando filas 2890 a 2899
Se corto el proceso del LLM por este motivo : 'Abstract' en el i

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.


Se corto el proceso del LLM por este motivo : 'Abstract' en el id 23805209
Procesando filas 3450 a 3459
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 32928965
Procesando filas 3460 a 3469
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 24086683
Procesando filas 3470 a 3479
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 26771402
Procesando filas 3480 a 3489
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 22421690
Procesando filas 3490 a 3499
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 25533217
Procesando filas 3500 a 3509
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 32562845
Procesando filas 3510 a 3519
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 28178111
Procesando filas 3520 a 3529
Se corto el proceso del LLM por este motivo : 'Abstract' en el id 25465668
Procesando filas 3530 a 3539
Se corto el proceso del LLM por este motivo : 'Abstract' en el i

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11068\3837796252.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.


Se corto el proceso del LLM por este motivo : 'Abstract' en el id 26372927
Shape final: (0, 7)


In [20]:
df_Cta.shape
df_Cta.head()

,Abstract,Title,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
22304240,The direct interrogation of fleeting intermedi...,NaN,None,None,None,NaN,4,The abstract does not contain any information...
21848430,Chlamydial infection of the lower genital trac...,NaN,None,None,None,NaN,4,The abstract does not contain any information...
22042092,No abstract available,NaN,None,None,None,NaN,4,The abstract does not contain any information...
22203235,A gram-negative obligate intracellular bacteri...,NaN,None,None,None,NaN,4,The abstract does not contain any information...
22222354,Ribonucleotide reductases (RNRs) are essential...,NaN,None,None,None,NaN,4,The abstract does not contain any information...


In [ ]:
df_Cta.shape

### 5) Evaluar 

In [67]:
evaluacion_score(df_Cta)

values=[1 1 0]
ai_opcion1
values=[1 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion1
values=[0 0 1]
ai_opcion2
values=[0 1 0]
ai_opcion1
values=[1 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[0 0 1]
ai_opcion2
values=[1 1 0]
ai_opcion1
values=[0 0 1]
ai_opcion3
values=[1 1 0]
ai_opcion0
values=[0 1 0]
ai_opcion3
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion0
values=[0 1 0]
ai_opcion3
values=[0 1 0]
ai_opcion3
values=[0 0 0]
ai_opcion3
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[0 1 0]
ai_opcion1
values=[1 0 0]
ai_opcion0
values=[1 0 0]
ai_opcion3
values=[1 1 

0.8287671232876712

# 6) Limpieza de Df

In [24]:
columns_excluded = ['1) Antimicrobial Resistance stain', 
                    '2) New treatment', 
                    '3) Immunization',  
                    'Human_summary'
                    ]

In [25]:
df_clean= df_Cta.drop(columns= columns_excluded)

In [26]:
df_clean

,Abstract,Title,ai_label,ai_summary
PMID,,,,
22304240,The direct interrogation of fleeting intermedi...,NaN,4,The abstract does not contain any information...
21848430,Chlamydial infection of the lower genital trac...,NaN,4,The abstract does not contain any information...
22042092,No abstract available,NaN,4,The abstract does not contain any information...
22203235,A gram-negative obligate intracellular bacteri...,NaN,4,The abstract does not contain any information...
22222354,Ribonucleotide reductases (RNRs) are essential...,NaN,4,The abstract does not contain any information...
...,...,...,...,...
36704557,Artificial tick feeding systems (ATFS) can be ...,NaN,4,The abstract does not contain any information...
36714306,"The sweet potato whitefly, Bemisia tabaci (Gen...",NaN,4,The abstract does not contain any information...
36726570,Zooplankton provides bacteria with a complex m...,NaN,4,The abstract does not contain any information...


# EDA

In [27]:
print("Se analizaron ", df_clean.shape[0], " papers.")

Se analizaron  24967  papers.


#### AI Label:

1) Antimicrobial Resistance stain
2) New treatment
3) Immunization
4) None

In [28]:
df_clean['ai_label'].value_counts()

ai_label
4    24967
Name: count, dtype: int64